In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# for every T'th token we want the average of all the previous tokens including the T'th
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# x[B, T] = mean{i <= t} x[b, i]
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]   
        xbow[b, t] = torch.mean(xprev, 0)

In [3]:
# matmul (to be more efficient)

# Create a lower-triangular matrix of 1s
wei = torch.tril(torch.ones(T, T))
# Divide each row by its sum (1, 2, 3, ..., T) to get weights for averaging
wei = wei / wei.sum(1, keepdim=True)
# Matrix multiply to compute the running averages
xbow2 = wei @ x       # (T, T) @ (B, T, C) -> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6)

True

In [4]:
# using softmax
# Why we use softmax + -inf instead of simple division:
# 1. Causality: Setting future tokens to -inf forces their softmax weight to 0 (e^-inf = 0),
#    preventing future information from leaking into the current prediction.
# 2. Bridge to Self-Attention: Right now `wei` is all zeros, so softmax gives equal weights (a simple average).
#    In full self-attention, `zeros` is replaced by data-dependent Query-Key affinity scores (`q @ k.T`).
#    Softmax normalizes those arbitrary scores into probabilities that sum to 1.

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))

wei = wei.masked_fill(tril == 0, float('-inf'))      #for all elements where tril is 0 it becomes -inf
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow2, xbow3)

True

In [ ]:
#self-attention
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# every single token emmits two vectors Query(what i am looking for) and Key(what i have), Value(if you find me interesting here is what i will give it to you)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   #(B, T, 16)
q = query(x) #(B, T, 16)
wei = q @ k.transpose(-2, -1) #* head_size ** -0.5     #(B, T, T)     each batch is processed independently they don't talk to each oher


tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))    # for decoder block, hides the future tokens, for encoder block we do not need this as we want all the tokens to talk to each other.
wei = F.softmax(wei, dim=1)
#out = wei @ x
v = value(x)
out = wei @ v  
out.shape

torch.Size([4, 8, 16])

In [ ]:
#we also add a root scaling
weiii = q @ k.transpose(-2, -1) * head_size ** -0.5
print(q.var(), k.var(), weiii.var()) # as you can see weiii variance is higher, so scaling is used just to control the variance at initialization


tensor(0.3084, grad_fn=<VarBackward0>) tensor(0.3291, grad_fn=<VarBackward0>) tensor(1.3439, grad_fn=<VarBackward0>)
